# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** order for the orders_json list, and one vendor for the vendors_json list

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [6]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
orders = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
# TODO: assert the row count and the quantity total
assert len(orders) == 4
assert orders['qty'].sum() == 7

In [7]:
orders.head()

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


I used record_path='lines' to make a row for each order, since this is where the order-specific data starts in the original orders_json. meta=['order', 'vendor_id'] allows me to include the order number and vendor id for each order, even though these are not unique and are included in front of 'lines'. Now, each row represents a specific item, rather than an order.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [9]:
# TODO
vendors = pd.json_normalize(vendors_json)
joined = orders.merge(vendors, on='vendor_id', how='left', indicator=True)
unmatched = joined[joined['_merge'] == 'left_only']
unmatched

,item,qty,order,vendor_id,name,zone,_merge
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


I turned vendors_json into a dataframe and left joined it to orders using the shared 'vendor_id' column. The vendor ID V-07 has no matching vendor. This vendor has 1 order, of 3 items, attached to it.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [10]:
# TODO
joined['name'] = joined['name'].fillna('Unknown Vendor')
joined['zone'] = joined['zone'].fillna('Unknown Zone')

I left the unmatched vendor in the table, and labeled its name and zone as unknown. I did this because the table is fairly small, so the row with the unknown vendor is a major portion of the data and should not be removed, because the impact on sales info is significant. Additionally, if there is a vendor that was missed, it would be important to figure this out rather than just discarding it.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [12]:
# TODO
sales_by_zone = joined.groupby('zone')['qty'].sum().sort_values(ascending=False)

print(sales_by_zone)

zone
A               4
Unknown Zone    3
Name: qty, dtype: int64


I grouped by zone and summed the item quantities for each grouping. Zone A sold the most items, with 4. Since I did keep the Unknown Zone, my zone total still adds up to 7 items.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [13]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
new_json = [{**order, 'lines': order.get('lines', [])} for order in ragged_json]
new_orders = pd.json_normalize(new_json, record_path='lines', meta=['order', 'vendor_id'])
new_orders['qty'] = new_orders['qty'].astype('Int64')

new_orders

,item,qty,order,vendor_id
0,Soda,2,4,V-01
1,Fries,<NA>,6,V-01


I made new_json as a json that sorts through the ragged one and remakes it using the original information, but produces an empty list for any order that does not contain a lines item. I then flattened it using the same method as previously in the notebook, and I converted the 'qty' column to integer so that a missing quantity will read as NA rather than 0.

### Q6 — Validate

In [14]:
assert len(orders) == 4
assert orders['qty'].sum() == 7
assert len(joined) == len(orders), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

1 row per line item is useful because it allows us to distinguish by item, rather than by order, as the original layout had. This is important if we have price per item and want to calculate revenue, because each row represents a single item and states its quantity. If each row represented an order, it would be much more difficult to calculate revenue because rows might contain multiple items, with multiple prices.

In Q5, missing quantities are very different from quantities of 0 because if you are trying to calculate revenue downstream, a quantity of 0 will obviously produce a revenue of 0, which would be correct. However, a missing quantity that conflates as 0 quantity will also produce a revenue of 0, which is not correct if the quantity is in fact more than 0. A quantity of 0 produces revenue of 0, but a missing quantity could represent any number of revenue, depending on what the true quantity is, so we cannot treat them the same.